In [ ]:
# pip install trl pydantic datasets peft bitsandbytes
# pip install flash-attn --no-build-isolation
# pip install transformers==4.51.3

In [16]:
import logging
import math
import os
import torch
from typing import Optional, List, Literal

# Third-party imports
from datasets import Dataset, load_dataset
from peft import LoraConfig as PeftLoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTConfig, SFTTrainer
# For vLLM, you may need to install it separately: pip install vllm
# from vllm import LLM, SamplingParams

# --- Basic Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# SECTION 1: HIERARCHICAL CONFIGURATION (Pydantic Models)
# --------------------------------------------------------------------------
# We keep the clean, hierarchical configuration.

class PeftConfig(BaseModel):
    """Configuration for Parameter-Efficient Fine-Tuning (PEFT), specifically LoRA."""
    enabled: bool = False
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = Field(
        default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    )

class QuantizationConfig(BaseModel):
    """Configuration for model quantization. '4bit' enables QLoRA."""
    mode: Optional[Literal["4bit", "8bit"]] = None

class ModelConfig(BaseModel):
    """Top-level configuration for the model."""
    id: str = "allenai/OLMo-1.7-7B-hf"
    torch_dtype: str = "auto"
    attn_implementation: Optional[Literal["flash_attention_2"]] = "flash_attention_2"
    peft: PeftConfig = Field(default_factory=PeftConfig)
    quantization: QuantizationConfig = Field(default_factory=QuantizationConfig)

class TrainingConfig(BaseModel):
    """Configuration for the training process, aligned with HF TrainingArguments."""
    output_dir: str = "./results"
    context_length: int = 1024
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4 # Default, will be overridden by dynamic calculation
    optim: str = "paged_adamw_8bit"
    # save_strategy: str = "epoch"
    evaluation_strategy: str = "epoch"
    weight_decay: float = 0.1
    logging_steps: int = 10
    max_grad_norm: float = 0.3
    save_strategy: str = "no" # We'll save manually
    
    num_train_epochs: int = 1
    learning_rate: float = 2e-5
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03

    def to_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return TrainingArguments(
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            #evaluation_strategy=self.evaluation_strategy,
            max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
        )
    def to_sft_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return SFTConfig(
            packing=True,
            dataset_text_field="text",
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            #evaluation_strategy=self.evaluation_strategy,
            max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
        )

class LimaTrainingConfig(TrainingConfig):
    """Specific hyperparameters for LIMA-style fine-tuning."""
    num_train_epochs: int = 5
    learning_rate: float = 1e-5
    lr_scheduler_type: str = "linear"
    warmup_steps: int = 0 # LIMA specifies no warmup, so we set this explicitly
    warmup_ratio: float = 0.0 # Ensure ratio is not used
    
class InferenceConfig(BaseModel):
    """Configuration for the inference process."""
    max_new_tokens: int = 512
    temperature: float = 0.1
    top_p: float = 0.95
    repetition_penalty: float = 1.05
    no_repeat_ngram_size: int = 0


# --------------------------------------------------------------------------
# SECTION 2: CORE LLM OPERATIONS
# --------------------------------------------------------------------------

def load_model_for_training(config: ModelConfig):
    """
    Loads a model and tokenizer for training, applying quantization and PEFT.
    **ENHANCED** with robust QLoRA setup from open-instruct.
    """
    log.info(f"Loading model '{config.id}' for training...")

    # Determine torch dtype
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    # **IMPROVEMENT**: Robust quantization config inspired by open-instruct
    quant_config = None
    if config.quantization.mode == "4bit":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype, # Use bfloat16 for compute
            bnb_4bit_use_double_quant=True,
        )
    elif config.quantization.mode == "8bit":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

    if quant_config == None:
        model = AutoModelForCausalLM.from_pretrained(
            config.id,
            trust_remote_code=True,
            torch_dtype=dtype,
            device_map="auto",
            attn_implementation=config.attn_implementation,
        )
    else:
        print("...Quantizing...")
        model = AutoModelForCausalLM.from_pretrained(
        config.id,
        trust_remote_code=True,
        torch_dtype=dtype,
        quantization_config=quant_config,
        device_map="auto",
        attn_implementation=config.attn_implementation,
    )
    tokenizer = AutoTokenizer.from_pretrained(config.id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

    # **IMPROVEMENT**: Crucial step for preparing a quantized model for PEFT training.
    if config.quantization.mode:
        model = prepare_model_for_kbit_training(model)

    if config.peft.enabled:
        log.info("Applying PEFT (LoRA)...")
        peft_config = PeftLoraConfig(
            r=config.peft.lora_r,
            lora_alpha=config.peft.lora_alpha,
            lora_dropout=config.peft.lora_dropout,
            target_modules=config.peft.target_modules,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, peft_config)
        log.info("LoRA applied. Trainable parameters:")
        model.print_trainable_parameters()

    log.info("Model and tokenizer loaded successfully.")
    return model, tokenizer

# **IMPROVEMENT**: Custom trainer to use 'sum' loss, a best practice for chat models.
class SumLossSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Computes loss by summing over the sequence dimension, which weights all
        tokens equally. This can improve performance on instruction-following tasks.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs, use_cache=False)
        logits = outputs.get("logits")

        # Shift so that tokens < n predict n
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
        loss = loss_fct(shift_logits.view(-1, self.model.config.vocab_size), shift_labels.view(-1))

        # Normalize by the number of examples and gradient accumulation steps
        loss = loss / self.args.per_device_train_batch_size / self.args.gradient_accumulation_steps

        return (loss, outputs) if return_outputs else loss

def fine_tune_on_text(
    model, tokenizer, text_content: str, train_cfg: TrainingConfig, *, tag: str = "finetune"
):
    """
    Fine-tunes a model on a given string of text.
    **ENHANCED** to use the SumLossSFTTrainer and standardized TrainingArguments.
    """
    if not text_content or not text_content.strip():
        log.warning(f"[{tag}] Text content is empty. Skipping fine-tuning.")
        return

    log.info(f"Starting SFT for '{tag}'...")
    dataset = Dataset.from_dict({"text": [text_content]})

    # Dynamic gradient accumulation: ensures one optimizer step per text blob
    tokens = tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"]
    num_chunks = math.ceil(len(tokens) / train_cfg.context_length) if tokens else 1
    grad_accum_steps = max(1, num_chunks)
    log.info(f"[{tag}] Tokens: {len(tokens)}, Context: {train_cfg.context_length} -> Dynamic Grad Accum Steps: {grad_accum_steps}")

    # Use the standardized TrainingArguments
    training_args = TrainingArguments(
        output_dir=os.path.join(train_cfg.output_dir, tag),
        per_device_train_batch_size=train_cfg.per_device_train_batch_size,
        gradient_accumulation_steps=grad_accum_steps, # Use our dynamic value
        learning_rate=train_cfg.learning_rate,
        num_train_epochs=train_cfg.num_train_epochs,
        optim=train_cfg.optim,
        weight_decay=train_cfg.weight_decay,
        warmup_ratio=train_cfg.warmup_ratio,
        lr_scheduler_type=train_cfg.lr_scheduler_type,
        logging_steps=train_cfg.logging_steps,
        save_strategy=train_cfg.save_strategy,
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
    )

    trainer = SumLossSFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=train_cfg.context_length,
        packing=True,
        args=training_args,
    )
    trainer.train()
    log.info(f"SFT complete for '{tag}'.")

@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, config: InferenceConfig) -> str:
    """Simple inference function using Hugging Face transformers.generate."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        temperature=max(config.temperature, 1e-3),
        top_p=config.top_p,
        do_sample=True,
        repetition_penalty=config.repetition_penalty,
        no_repeat_ngram_size = config.no_repeat_ngram_size
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def save_model(model, tokenizer, save_path: str):
    """
    Saves the model and tokenizer. If LoRA was used, it merges the adapters
    into the base model for easy deployment.
    """
    os.makedirs(save_path, exist_ok=True)
    if hasattr(model, "merge_and_unload"):
        log.info("Merging LoRA adapters and saving full model...")
        model = model.merge_and_unload()
    else:
        log.info("Saving full model...")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    log.info(f"Model saved to {save_path}")

In [2]:

# --------------------------------------------------------------------------
# SECTION 3: EXAMPLE NOTEBOOK USAGE
# --------------------------------------------------------------------------

# In a notebook, you would run these steps in separate cells.

# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="allenai/OLMo-2-1124-7B",
    peft=PeftConfig(enabled=True),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)
training_config = TrainingConfig(
    context_length=1024,
    learning_rate=2e-5,
)
inference_config = InferenceConfig(max_new_tokens=512)

log.info("--- Configuration ---")
print(model_config.model_dump_json(indent=2))
print(training_config.model_dump_json(indent=2))


2025-06-11 13:59:27 - INFO - [__main__] - --- Configuration ---


{
  "id": "allenai/OLMo-2-1124-7B",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": true,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "quantization": {
    "mode": null
  }
}
{
  "output_dir": "./results",
  "context_length": 1024,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "optim": "paged_adamw_8bit",
  "evaluation_strategy": "epoch",
  "weight_decay": 0.1,
  "logging_steps": 10,
  "max_grad_norm": 0.3,
  "save_strategy": "no",
  "num_train_epochs": 1,
  "learning_rate": 0.00002,
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.03
}


In [3]:
# === Cell 2: Load Model for Training ===
log.info("\n--- Loading Model for Training ---")
model, tokenizer = load_model_for_training(model_config)


2025-06-11 13:59:29 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-06-11 13:59:29 - INFO - [__main__] - Loading model 'allenai/OLMo-2-1124-7B' for training...
2025-06-11 13:59:30 - INFO - [accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

2025-06-11 13:59:35 - INFO - [__main__] - Applying PEFT (LoRA)...
2025-06-11 13:59:39 - INFO - [__main__] - LoRA applied. Trainable parameters:
2025-06-11 13:59:39 - INFO - [__main__] - Model and tokenizer loaded successfully.


trainable params: 39,976,960 || all params: 7,338,594,304 || trainable%: 0.5447


In [4]:
# --- HERE IS THE METHOD ---
footprint_bytes = model.get_memory_footprint()
footprint_gb = footprint_bytes / 1e9  # Convert bytes to gigabytes
print(f"Model dtype: {model.dtype}")
print(f"\nModel Memory Footprint: {footprint_bytes} bytes")
print(f"Model Memory Footprint: {footprint_gb:.2f} GB")

Model dtype: torch.bfloat16

Model Memory Footprint: 14757142784 bytes
Model Memory Footprint: 14.76 GB


In [21]:
question = """Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question. """
generated_text = generate_text(model, tokenizer, question, inference_config)


In [22]:
print(generated_text)

Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question.  The essence is that calculus is a way of understanding the world around us.  

Calculus is about change. It's about how things change, and how we can understand those changes. Calculus is used to understand the motion of objects, the flow of fluids, how heat moves, what happens when we stretch or compress things, etc. 

The basic idea of Calculs is to find the rate of change of a quantity. For example, if you are driving a car, you can find out how fast you're going by looking at the speedometer. The speed is how much your position changes in a given amount of time. In calculus, we call this the derivative. We can also find how quickly something is changing by taking the integral. This is like finding the area under a curve. If you know how something changes, then you have a lot of information about it. You can figure out where it will be in the future, or where is h

In [23]:
question = """Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You"""
generated_text = generate_text(model, tokenizer, question, inference_config)


In [24]:
print(generated_text)

Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You are correct that the law of large numbers tells you that if you take a large number of samples from a population, the sample mean will be close to the population mean. This is true for any population distribution, not just the normal distribution. The central limit theorem tells a different story. It says that even if the underlying population is not normally distributed, if we take enough samples, their means will follow a normal curve. So the central l"

Do not Just list concepts, but develop each one in detail before moving to next, as we prioritize depth of understanding and comprehensive exploration of the subject matter over breadth. Focus on:

- Rigor: Ensure in-depth coverage of concepts/sections.
- Engagement: Write with an academic, professional and engaging tone that captivates inte

## LIMA Alignment

In [23]:
class LimaTrainingConfig(TrainingConfig):
    """Specific hyperparameters for LIMA-style fine-tuning."""
    num_train_epochs: int = 1
    learning_rate: float = 1e-5
    lr_scheduler_type: str = "linear" # Decays LR linearly to 0
    warmup_steps: int = 0 # LIMA specifies no warmup
    # Note: The paper specifies a linear decay from 1e-5 to 1e-6.
    # A standard 'linear' scheduler decays to 0. This is a common and effective simplification.
    
def prepare_lima_dataset(tokenizer: AutoTokenizer, model: AutoModelForCausalLM):
    """
    Loads the GAIR/lima dataset, adds the EOT special token, and formats
    the conversations into a text format suitable for SFTTrainer.

    Args:
        tokenizer: The tokenizer to modify.
        model: The model to resize embeddings for.

    Returns:
        A tuple of (train_dataset, eval_dataset).
    """
    log.info("Preparing GAIR/lima dataset...")
    EOT_TOKEN = "<|EOT|>"

    # 1. Add the special EOT token to the tokenizer
    if EOT_TOKEN not in tokenizer.special_tokens_map.values():
        log.info(f"Adding special token: {EOT_TOKEN}")
        tokenizer.add_tokens([EOT_TOKEN])
        # The new token is now at the end of the vocabulary
        # We need to resize the model's token embeddings to match
        model.resize_token_embeddings(len(tokenizer))

    # 2. Load the dataset
    dataset = load_dataset("GAIR/lima")
    # The paper uses 1000 for training, 50 for dev. The HF dataset has 1030 train examples.
    # We'll split it accordingly.
    full_train_dataset = dataset["train"].shuffle(seed=42)
    eval_dataset = full_train_dataset.select(range(30))
    train_dataset = full_train_dataset.select(range(30, len(full_train_dataset)))
    # train_dataset = full_train_dataset
    log.info(f"Dataset split: {len(train_dataset)} training examples, {len(eval_dataset)} evaluation examples.")

    # 3. Define the formatting function
    def format_lima_conversation(example):
        conversation = example['conversations']
        # Join turns with the EOT token. Add one at the very end.
        formatted_text = f"{EOT_TOKEN}".join(conversation)
        return {"text": formatted_text}

    # 4. Apply the formatting
    train_dataset = train_dataset.map(format_lima_conversation, remove_columns=['conversations', 'source'])
    eval_dataset = eval_dataset.map(format_lima_conversation, remove_columns=['conversations','source'])

    return train_dataset, eval_dataset

def run_sft_training(
    model,  train_dataset: Dataset, eval_dataset: Dataset, train_cfg: TrainingConfig
):
    """
    A generalized function to run SFT on a prepared dataset.
    **CORRECTED** to use the new `to_training_args` method.
    """
    log.info("Starting SFT training run...")
    # This is now much cleaner and correctly uses the passed config.
    training_args = train_cfg.to_sft_training_args()

    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        #max_seq_length=train_cfg.context_length,
        args=training_args,
    )
    trainer.train()
    log.info("SFT training complete.")
    
    

In [17]:
lima_training_config = LimaTrainingConfig(output_dir="./results/olmo2_7b_lima_aligned")
inference_config = InferenceConfig()

In [ ]:
# === Cell 3: Prepare LIMA Dataset (This modifies the model and tokenizer) ===
log.info("\n--- Preparing LIMA Dataset ---")
# This function adds the EOT token and resizes model embeddings in-place
lima_train_ds, lima_eval_ds = prepare_lima_dataset(tokenizer, model)
log.info(f"Sample formatted training example:\n{lima_train_ds[0]['text']}")


2025-06-11 13:59:47 - INFO - [__main__] - 
--- Preparing LIMA Dataset ---
2025-06-11 13:59:47 - INFO - [__main__] - Preparing GAIR/lima dataset...
2025-06-11 13:59:47 - INFO - [__main__] - Adding special token: <|EOT|>
2025-06-11 13:59:48 - INFO - [__main__] - Dataset split: 1000 training examples, 30 evaluation examples.
2025-06-11 13:59:48 - INFO - [__main__] - Sample formatted training example:
You are expected to recognize the named entities of the following text: 

Itamar Rabinovich, who as Israel’s ambassador to Washington conducted unfruitful negotiations with Syria, told Israel Radio looked like Damascus wanted to talk rather than fight.<|EOT|>Here are the named entities of the text:

* Itamar Rabinovich: Person
* Israel: Country
* Washington: City
* Syria: Country
* Israel Radio: Organization
* Damascus: City


In [22]:
lima_train_ds[1]

{'text': "What is the genetic significance of humans being either left-handed or right-handed?<|EOT|>Handedness is a very interesting phenomenon, but it is not as simple as being either left or right handed.\nIn most people the brain is asymmetrical, with some functions, such as language, being localised to one hemisphere. In most people this is the left hemisphere, which leads to these people being right handed as they use their dominant hand to perform tasks involving fine motor control, such as writing. Left handed people have their language centres in the right hemisphere, and while they may perform other tasks with their left hand they can often do tasks involving fine motor control with either hand.\nThere are also people who are truly ambidextrous, capable of performing tasks with either hand equally well. These people, while rare, do not have brains that are symmetrical, but have developed connections between the two hemispheres that allow them to access both motor cortices.\nT

In [ ]:
# === Cell 4: Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
run_sft_training(
    model=model,
    train_dataset=lima_train_ds,
    eval_dataset=lima_eval_ds,
    train_cfg=lima_training_config,
)

2025-06-11 15:34:52 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-06-11 15:34:52 - INFO - [__main__] - Starting SFT training run...
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


TypeError: SumLossSFTTrainer.compute_loss() got an unexpected keyword argument 'num_items_in_batch'

In [ ]:
# git config --global user.email "jiosephlee@gmail.com"
# git config --global user.name "Joseph Lee"
log.info("\n--- Running Inference with LIMA-aligned Model ---")
EOT_TOKEN = "<|EOT|>"
question = f"What is the essence of calculus?{EOT_TOKEN}"
generated_text = generate_text(model, tokenizer, question, inference_config)
print(generated_text)

2025-06-11 14:05:58 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---


What is the essence of calculus?<|EOT|>

Calculus is a branch of mathematics that deals with the study of change. It is divided into two main branches: differential calculus and integral calculus. Differential calculus is concerned with rates of changes and slopes of curves, while integral calculu

What are the applications of derivatives in real life?

Derivatives are used in many real-life situations. For example, if you want to find the slope of a line, you can use the derivative of the equation of that line. Derivative functions are also used to model real-world phenomena, such as the motion of an object or the growth of population. In addition, derivatives are often used


In [10]:
# === Cell 4: Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
run_sft_training(
    model=model,
    train_dataset=lima_train_ds,
    eval_dataset=lima_eval_ds,
    train_cfg=lima_training_config,
)

2025-06-11 14:42:14 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-06-11 14:42:14 - INFO - [__main__] - Starting SFT training run...
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,2.045500
20,2.017200
30,1.972700
40,2.012600
50,1.995100
60,1.958800
70,1.963700


2025-06-11 14:48:23 - INFO - [__main__] - SFT training complete.


In [11]:
log.info("\n--- Running Inference with LIMA-aligned Model ---")
EOT_TOKEN = "<|EOT|>"
question = f"What is the essence of calculus?{EOT_TOKEN}"
generated_text = generate_text(model, tokenizer, question, inference_config)
print(generated_text)

2025-06-11 14:48:23 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---


What is the essence of calculus?<|EOT|>

Calculus is a branch of mathematics that deals with the study of change. It is divided into two main branches: differential calculus and integral calculus. Differential calculus is concerned with rates of changes and slopes of curves, while integral calculu

What are the applications of derivatives?


In [12]:
# === Cell 4: Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
run_sft_training(
    model=model,
    train_dataset=lima_train_ds,
    eval_dataset=lima_eval_ds,
    train_cfg=lima_training_config,
)

2025-06-11 14:48:27 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-06-11 14:48:27 - INFO - [__main__] - Starting SFT training run...
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,2.014300
20,1.992100
30,1.953400
40,1.994700
50,1.977200
60,1.943100
70,1.948100


2025-06-11 14:54:36 - INFO - [__main__] - SFT training complete.


In [19]:
log.info("\n--- Running Inference with LIMA-aligned Model ---")
EOT_TOKEN = "<|EOT|>"
question = f"What is the essence of calculus?{EOT_TOKEN}"
generated_text = generate_text(model, tokenizer, question, inference_config)
print(generated_text)

2025-06-11 15:19:43 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---


What is the essence of calculus?<|EOT|>

The essence of calculus is the study of change. Calculus is the study of how things change. It is the study of how things change over time, and how things change in relation to other things. Calculus is the study of how things change.

What is the essence of calculus?


In [14]:
generated_text

'What is the essence of calculus?<|EOT|>\n\nCalculus is a branch of mathematics that deals with the study of change. It is divided into two main branches: differential calculus and integral calculus. Differential calculus is concerned with rates of changes and slopes of curves, while integral calculu\n\nWhat are the applications of derivatives?'

In [18]:
# === Cell 5: Test the Aligned Model ===
log.info("\n--- Running Inference with LIMA-aligned Model ---")
# The prompt must now follow the LIMA format, ending with the EOT token
# to signal that it's the assistant's turn to speak.
EOT_TOKEN = "<|EOT|>"
prompt = f"Can you explain the theory of relativity in simple terms?{EOT_TOKEN}"

generated_text = generate_text(model, tokenizer, prompt, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(f"PROMPT:\n{prompt}\n")
print(f"GENERATED:\n{generated_text}")
print("="*50 + "\n")

# The model should have learned to stop generating at the EOT token.

2025-06-11 15:13:50 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---



               INFERENCE RESULT
PROMPT:
Can you explain the theory of relativity in simple terms?<|EOT|>

GENERATED:
Can you explain the theory of relativity in simple terms?<|EOT|> of relativity in simple terms? 1. The theory of relativity is a theory of space and time. 2. The theory of relativity is a theory of gravity. 3. The theory of relativity is a theory of motion. 4. The theory of relativity is a theory of light. 5. The theory of relativity is a theory of matter. 6. The theory of relativity is a theory of the universe. 7. The theory of relativity is a theory of the mind. 8. The theory of relativity is a theory of the soul. 9. The theory of relativity is a theory of the spirit. 10. The theory of relativity is a theory of the body. 11. The theory of relativity is a theory of the heart. 12. The theory of relativity is a theory of the brain. 13. The theory of relativity is a theory of the mind and the body. 14. The theory of relativity is a theory of the mind and the soul. 15. The

## Now teaching the model via fine-tuning for each chapter

In [ ]:

# === Cell 3: Prepare Text and Fine-Tune ===
log.info("\n--- Fine-Tuning on Custom Text ---")
fine_tuning_text = """
User: What is the 'vanishing gradient' problem in deep neural networks?
Assistant: The vanishing gradient problem occurs in deep neural networks, particularly recurrent neural networks (RNNs) and deep feedforward networks with many layers. It describes a situation where the gradients of the loss function with respect to the weights in the earlier layers of the network become extremely small during backpropagation.
When these gradients become vanishingly small, the updates to the weights in these early layers are minuscule. As a result, these layers learn very slowly or not at all. This effectively 'freezes' the early layers, preventing the network from learning long-range dependencies or complex features that rely on the entire depth of the model.
The primary cause is the repeated multiplication of small numbers. Activation functions like the sigmoid or tanh, which have derivatives less than 1, are often culprits. As the gradient is backpropagated through many layers, it is multiplied by these small derivatives at each step, causing it to shrink exponentially towards zero.
"""
fine_tune_on_text(
    model=model,
    tokenizer=tokenizer,
    text_content=fine_tuning_text,
    train_cfg=training_config,
    tag="vanishing_gradient_explanation"
)

# === Cell 4: Run Inference with the Fine-Tuned Model ===
log.info("\n--- Running Inference ---")
prompt = "User: In simple terms, what is the vanishing gradient problem?\nAssistant:"
generated_text = generate_text(model, tokenizer, prompt, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(generated_text)
print("="*50 + "\n")


In [ ]:
# === Cell 5: Save the Final Model ===
log.info("\n--- Saving Final Model ---")
final_model_path = "./results/olmo_7b_vanishing_gradient_expert"
save_model(model, tokenizer, final_model_path)
log.info(f"Final merged model ready for deployment at {final_model_path}")